# Castle5000: làm giàu dữ liệu trên Kaggle / Colab

Notebook này chạy mọi bước cần GPU của stage `adl` và `enrich` rồi đóng gói kết quả để tải về máy.

**Chuẩn bị (một lần):**

1. Ở máy local, trong `labeling/`: `python main.py bundle code` tạo `labeling_code.zip`.
2. Đưa lên nền tảng:
   - **Kaggle:** tạo một Dataset chứa `labeling_code.zip`, một Dataset chứa thư mục ảnh; gắn cả hai vào notebook. Bật **GPU** và **Internet** trong Settings.
   - **Colab:** chép `labeling_code.zip` và thư mục ảnh (hoặc file zip ảnh) vào Google Drive. Chọn Runtime > Change runtime type > GPU.
3. Sửa ô **Cấu hình** bên dưới nếu notebook không tự tìm thấy đường dẫn.

Chạy lại notebook sau khi phiên bị ngắt sẽ tiếp tục từ chỗ dở, miễn là thư mục `WORK` còn (Kaggle: bật *Persistence*; Colab: `WORK` nằm trên Drive).

## 1. Cấu hình

In [ ]:
import os, sys, glob, shutil, zipfile, subprocess
from pathlib import Path

ON_KAGGLE = os.path.isdir("/kaggle/working")
ON_COLAB = not ON_KAGGLE and ("google.colab" in sys.modules or os.path.isdir("/content"))

# --- Sửa ba dòng này nếu cần. None = tự tìm. ---
CODE_ZIP = None      # ví dụ "/kaggle/input/castle-code/labeling_code.zip"
IMAGES_DIR = None    # ví dụ "/kaggle/input/castle5000/images"
IMAGES_ZIP = None    # (Colab) file zip ảnh trên Drive, sẽ được giải nén ra đĩa cục bộ cho nhanh

if ON_KAGGLE:
    # WORK nằm ngoài /kaggle/working để hàng chục nghìn file crop không thành "output" của notebook
    # (tải về rất chậm). Chỉ file zip kết quả được ghi vào /kaggle/working. Muốn giữ tiến độ giữa các
    # phiên tương tác có bật Persistence thì đổi thành Path("/kaggle/working/castle").
    SEARCH, WORK = "/kaggle/input", Path("/kaggle/temp/castle")
elif ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    SEARCH, WORK = "/content/drive/MyDrive", Path("/content/drive/MyDrive/castle_work")
else:
    SEARCH, WORK = os.getcwd(), Path(os.getcwd()) / "castle_work"

WORK.mkdir(parents=True, exist_ok=True)
CODE_DIR = Path("/kaggle/working/labeling" if ON_KAGGLE else "/content/labeling" if ON_COLAB else WORK / "labeling")
print("Kaggle:", ON_KAGGLE, "| Colab:", ON_COLAB, "| WORK:", WORK)

## 2. Mã nguồn

In [ ]:
if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)          # luôn lấy bản mã nguồn mới nhất; dữ liệu nằm ở WORK nên không mất
if CODE_ZIP is None:
    found = sorted(glob.glob(f"{SEARCH}/**/labeling_code.zip", recursive=True))
    CODE_ZIP = found[-1] if found else None
if CODE_ZIP:
    with zipfile.ZipFile(CODE_ZIP) as z:
        z.extractall(CODE_DIR.parent)
    source = CODE_ZIP
else:
    # Kaggle tự giải nén file zip trong Dataset: tìm thư mục labeling/ đã bung sẵn.
    found = sorted(glob.glob(f"{SEARCH}/**/labeling/main.py", recursive=True))
    assert found, f"Không thấy labeling_code.zip hay labeling/main.py dưới {SEARCH}. Đặt CODE_ZIP bằng tay."
    source = os.path.dirname(found[-1])
    shutil.copytree(source, CODE_DIR)
assert (CODE_DIR / "main.py").exists(), f"Không có labeling/main.py trong {source}"
print("Mã nguồn:", CODE_DIR, "<-", source)

## 3. Ảnh

In [ ]:
def find_image_dir(root, min_images=5):
    """Thư mục có nhiều ảnh .jpg nhất dưới root."""
    best, best_n = None, 0
    for d, _, files in os.walk(root):
        n = sum(f.lower().endswith((".jpg", ".jpeg", ".png")) for f in files)
        if n > best_n:
            best, best_n = d, n
    return (best, best_n) if best_n >= min_images else (None, best_n)

if IMAGES_DIR is None and IMAGES_ZIP:
    local = Path("/content/castle_images" if ON_COLAB else WORK / "images")
    if not local.exists():
        with zipfile.ZipFile(IMAGES_ZIP) as z:
            z.extractall(local)
    IMAGES_DIR, n = find_image_dir(local)
elif IMAGES_DIR is None:
    IMAGES_DIR, n = find_image_dir(SEARCH)
assert IMAGES_DIR, "Không tìm thấy thư mục ảnh. Đặt IMAGES_DIR bằng tay."
n = sum(f.lower().endswith((".jpg", ".jpeg", ".png")) for f in os.listdir(IMAGES_DIR))
print(f"Ảnh: {IMAGES_DIR} ({n} file), ví dụ {sorted(os.listdir(IMAGES_DIR))[:3]}")

## 4. Thư viện và GPU

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_DIR / "requirements-cloud.txt")], check=True)

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(torch.cuda.get_device_name(0), f"- trống {free / 2**30:.1f} / {total / 2**30:.1f} GB")
else:
    print("CẢNH BÁO: không có GPU. Pipeline vẫn chạy nhưng sẽ mất nhiều giờ.")

## 5. Chạy pipeline

Ba biến môi trường dưới đây trỏ mọi đầu ra vào `WORK`. Lệnh `pipeline` chạy lần lượt và bỏ qua bước đã xong:

1. `adl embed`: embedding SigLIP2 của cả ảnh + trừ nền
2. `enrich build`: dò người (RT-DETR), cắt crop người và crop vùng, mã hoá
3. `init` + `suggest` + `export` cho ảnh, người và từng vùng
4. `enrich merge`: một bản ghi JSON cho mỗi ảnh

Lần đầu sẽ tải model về (khoảng 5 GB). Trên RTX 4050 6 GB cả pipeline mất khoảng 50 phút cho 5000 ảnh.

In [ ]:
os.environ["LABELING_INPUT_IMAGES"] = str(IMAGES_DIR)
os.environ["LABELING_OUTPUT_ROOT"] = str(WORK)
os.environ["LABELING_DATA_ROOT"] = str(WORK / "data")
os.environ["PYTHONIOENCODING"] = "utf-8"

def run(*args):
    """Chạy một lệnh của main.py, in log trực tiếp, dừng notebook nếu lỗi."""
    cmd = [sys.executable, "main.py", *args]
    print("$", " ".join(cmd))
    proc = subprocess.Popen(cmd, cwd=CODE_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    for line in proc.stdout:
        print(line, end="")
    assert proc.wait() == 0, f"Lệnh thất bại: {' '.join(args)}"

In [ ]:
# Thử nhanh trên 40 ảnh trước (vài phút). Bỏ comment để chạy; kết quả thử nằm riêng ở WORK/smoke.
# os.environ["LABELING_OUTPUT_ROOT"], os.environ["LABELING_DATA_ROOT"] = str(WORK / "smoke"), str(WORK / "smoke" / "data")
# run("pipeline", "--limit", "40")
# os.environ["LABELING_OUTPUT_ROOT"], os.environ["LABELING_DATA_ROOT"] = str(WORK), str(WORK / "data")

In [ ]:
run("pipeline")

## 6. Đóng gói kết quả

In [ ]:
RESULT_ZIP = (Path("/kaggle/working") if ON_KAGGLE else WORK) / "castle_results.zip"
run("bundle", "pack", str(RESULT_ZIP))
print("Tải file này về:", RESULT_ZIP, f"({RESULT_ZIP.stat().st_size / 1e6:.0f} MB)")
if ON_KAGGLE:
    from IPython.display import FileLink, display
    display(FileLink(str(RESULT_ZIP)))   # hoặc lấy ở tab Output sau khi Save Version

Ở máy local, trong `labeling/`:

```bash
python main.py bundle unpack duong_dan/castle_results.zip
```

Lệnh này giải nén vào `data/` và `output_jsons/`, rồi dựng lại các trang duyệt `review_*.html` theo đường dẫn của máy local. Profile nào ở máy local đã có nhãn người duyệt thì được giữ nguyên.

## 7. Nạp kết quả ngay trong notebook

In [ ]:
import json
import numpy as np

DATA, JSONS = WORK / "data", WORK / "output_jsons"

# Embedding cả ảnh (đã cộng vector vùng có người) và crop người
image_ids = [Path(n).stem for n in json.load(open(DATA / "adl" / "ids.json"))]
image_emb = np.load(DATA / "adl" / "embeddings.npy")
person_ids = [Path(n).stem for n in json.load(open(DATA / "enrich" / "people" / "ids.json"))]
person_emb = np.load(DATA / "enrich" / "people" / "embeddings.npy")
people = json.load(open(DATA / "enrich" / "people" / "people.json"))
print("ảnh:", image_emb.shape, "| người:", person_emb.shape)

# Bản ghi gộp của một ảnh
record = json.load(open(sorted((JSONS / "enriched").glob("*.json"))[len(image_ids) // 2], encoding="utf-8"))
print(json.dumps(record, ensure_ascii=False, indent=1)[:1500])

In [ ]:
# Tìm crop người theo câu tiếng Anh (cùng không gian vector với ảnh)
import open_clip, torch

sys.path.insert(0, str(CODE_DIR))
from modules.utils import load_config
clip_cfg = load_config(CODE_DIR / "config.yaml")["adl"]["clip"]
device = "cuda" if torch.cuda.is_available() else "cpu"
model, _, _ = open_clip.create_model_and_transforms(clip_cfg["model"], pretrained=clip_cfg["pretrained"], device=device)
model = model.eval().half() if device == "cuda" else model.eval()
tokenizer = open_clip.get_tokenizer(clip_cfg["model"])

def search_people(query, k=8):
    with torch.no_grad():
        q = model.encode_text(tokenizer([query]).to(device)).float()
    q = (q / q.norm(dim=-1, keepdim=True)).cpu().numpy()[0]
    top = np.argsort(-(person_emb @ q))[:k]
    return [(person_ids[i], people[person_ids[i]]["zone"], float(person_emb[i] @ q)) for i in top]

from IPython.display import Image as show, display
for pid, zone, score in search_people("a person opening a refrigerator door"):
    print(pid, zone, round(score, 3))
    display(show(filename=str(DATA / "enrich" / "people" / "crops" / f"{pid}.jpg"), width=160))